In [1]:
# Cell 1: Imports and Configuration
# -*- coding: utf-8 -*-
"""
DLR-PCVW v28.1: Distribution-aware Lesion Representation (STABILIZED)
FIX:
1. REMOVED unstable Linear Projection (3.2M params).
2. ADDED learnable per-channel variance modulation (w * sigma).
3. Variance injection is bounded by softplus(-3.0) to prevent explosion.
Target: 80%+ Stable (Clean representation, robust for thesis)
"""

import os, random, time, json, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score
import seaborn as sns
from collections import defaultdict

warnings.filterwarnings('ignore')
ImageFile.LOAD_TRUNCATED_IMAGES = True
torch.backends.cudnn.benchmark = True

# ============================================================
# CONFIGURATION
# ============================================================
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT   = r"D:\My Folder\PUC\Thesis\Archive\Acne04\acne_1024"

INPUT_SIZE     = 224
N_WAY          = 4
TRAIN_K        = 5
Q_QUERY        = 10
TRAIN_EPISODES = 1200
TEST_EPISODES  = 300
SEEDS          = list(range(5))
SPLIT_SEED     = 42
SPLIT_RATIO    = 0.7
K_HEADS        = 4

# MHOCP & DLR Weights
TEMPERATURE        = 16.0
CONSISTENCY_LAMBDA = 0.15
LDL_LAMBDA         = 0.4
WARMUP             = 400

WEIGHT_DECAY = 0.008
LABEL_SMOOTHING = 0.05
LR_BASE = 2e-4

PLOT_DIR = f"Final-dlr_pcvw_{INPUT_SIZE}px"
os.makedirs(PLOT_DIR, exist_ok=True)

train_transform = transforms.Compose([
    transforms.Resize(int(INPUT_SIZE * 1.14)), transforms.CenterCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(p=0.5), transforms.ColorJitter(0.08, 0.08, 0.08),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
test_transform = transforms.Compose([
    transforms.Resize(int(INPUT_SIZE * 1.14)), transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [2]:
# Cell 2: Data Splits and Cache
def make_splits(root, split_ratio=0.7, seed=42):
    py, np_s = random.getstate(), np.random.get_state()
    random.seed(seed); splits = {}
    for cid, cls in enumerate(sorted(os.listdir(root))):
        d = os.path.join(root, cls)
        if not os.path.isdir(d): continue
        imgs = [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith((".jpg",".png",".jpeg"))]
        random.shuffle(imgs); sp = int(len(imgs)*split_ratio)
        splits[cid] = {'train': imgs[:sp], 'test': imgs[sp:], 'name': cls, 'total': len(imgs)}
    random.setstate(py); np.random.set_state(np_s)
    for cid in splits: assert not (set(splits[cid]['train']) & set(splits[cid]['test']))
    print(f"[Splits] seed={seed} | ratio={split_ratio}", flush=True)
    return splits

def build_cache(splits, split="train", use_aug=False):
    t = train_transform if (split == "train" and use_aug) else test_transform
    print(f"[Cache] Loading {split} images...", flush=True)
    t0 = time.time(); cache = {}
    for cid in splits:
        for p in splits[cid][split]: cache[p] = t(Image.open(p).convert("RGB"))
    print(f"[Cache] {len(cache)} imgs | {sum(v.numel()*4 for v in cache.values())/1e6:.0f}MB | {time.time()-t0:.1f}s\n", flush=True)
    return cache

class FewShotDataset:
    def __init__(self, splits, split="train", cache=None, use_aug=False):
        self.data = {c: splits[c][split] for c in splits}
        self.cache = cache; self.transform = train_transform if use_aug else test_transform
    def _l(self, p):
        return self.cache[p] if (self.cache and p in self.cache) else self.transform(Image.open(p).convert("RGB"))
    def sample_episode(self, n_way, k_shot, q_query):
        classes = sorted(random.sample(list(self.data.keys()), n_way))
        sx, sy, qx, qy = [], [], [], []
        for ny, c in enumerate(classes):
            imgs = random.sample(self.data[c], k_shot + q_query)
            for p in imgs[:k_shot]: sx.append(self._l(p)); sy.append(ny)
            for p in imgs[k_shot:]: qx.append(self._l(p)); qy.append(ny)
        return torch.stack(sx), torch.tensor(sy), torch.stack(qx), torch.tensor(qy)

In [3]:
# Cell 3: Model Architecture (DLR-PCVW)
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = base.features
        for m in self.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.weight.requires_grad = False; m.bias.requires_grad = False

    def forward(self, x):
        self.features.eval() 
        return self.features(x)  # (B, 1280, 7, 7)

class DLR_Pool(nn.Module):
    def __init__(self, in_dim=1280, hidden_dim=256, n_classes=4):
        super().__init__()
        # 1. FiLM Calibration (Preserves MHOCP stability)
        self.stat_proj = nn.Sequential(
            nn.Linear(in_dim*2, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, in_dim*2)
        )
        
        # 2. Lesion Scorer (Soft response map)
        self.lesion_scorer = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )
        
        # 🔑 FIX: Replace Large Projection with Learnable Variance Weight
        self.var_weight = nn.Parameter(torch.full((in_dim,), -3.0))
        
        # 3. Consistency Heads
        self.head_predictors = nn.ModuleList([
            nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, n_classes))
            for _ in range(K_HEADS)
        ])
        for p in self.head_predictors:
            nn.init.xavier_uniform_(p[0].weight); nn.init.zeros_(p[0].bias)
            nn.init.xavier_uniform_(p[2].weight); nn.init.zeros_(p[2].bias)

    def forward(self, fm, return_vis=False):
        B, C, H, W = fm.shape
        
        # --- Step 1: FiLM Calibration ---
        stats = torch.cat([fm.mean(dim=[2,3]), fm.var(dim=[2,3])], dim=1)
        film = self.stat_proj(stats)
        gamma = film[:, :C].view(B, C, 1, 1); beta = film[:, C:].view(B, C, 1, 1)
        fm_calib = fm * (1 + gamma) + beta
        
        # --- Step 2: Extract Patch Tokens ---
        tokens = fm_calib.view(B, C, -1).permute(0, 2, 1)  # [B, 49, C]
        
        # --- Step 3: Soft Lesion Response ---
        scores = torch.sigmoid(self.lesion_scorer(tokens))  # [B, 49, 1]
        score_sum = scores.sum(dim=1) + 1e-6  # [B, 1]
        
        # --- Step 4: Weighted Mean (Lesion Intensity) ---
        mu = (tokens * scores).sum(dim=1) / score_sum  # [B, C]
        
        # --- Step 5: Weighted Variance (Lesion Spread/Density) ---
        diff_sq = (tokens - mu.unsqueeze(1)) ** 2  # [B, 49, C]
        var = (diff_sq * scores).sum(dim=1) / score_sum  # [B, C]
        
        # --- Step 6: STABLE FUSION (No Projection Layer) ---
        weight = F.softplus(self.var_weight)  # [C]
        feat = mu + weight * var  # [B, C]
        
        # Normalize for cosine similarity
        feat = F.normalize(feat, p=2, dim=1)
        
        # Consistency heads
        head_logits = [head(feat) for head in self.head_predictors]
        
        if return_vis:
            return feat, head_logits, scores.view(B, -1), mu, var, weight
        return feat, head_logits

class DLR_ProtoNet(nn.Module):
    def __init__(self, feat_dim=1280, n_classes=4):
        super().__init__()
        self.extractor = FeatureExtractor()
        self.dlr_pool = DLR_Pool(feat_dim, 256, n_classes)
        
        self.register_buffer('running_mean', torch.zeros(n_classes, feat_dim))
        self.register_buffer('running_count', torch.zeros(n_classes))
        self.semantic_step = nn.Parameter(torch.zeros(n_classes, feat_dim))

    def update_ema(self, s_feat, sy):
        for c in torch.unique(sy, sorted=True):
            mask = (sy == c); gc = c.item()
            bm = s_feat[mask].mean(0).detach()
            if self.running_count[gc] == 0: self.running_mean[gc] = bm
            else: self.running_mean[gc] = 0.99 * self.running_mean[gc] + 0.01 * bm
            self.running_count[gc] += 1

    def calibrate(self, raw_protos, sy):
        classes = torch.unique(sy, sorted=True); cal = []
        for ny, c in enumerate(classes):
            gc = c.item(); raw = raw_protos[ny]
            if self.running_count[gc] > 5:
                cal.append(0.75 * raw + 0.25 * self.running_mean[gc])
            else: cal.append(raw)
        return torch.stack(cal)

    def forward(self, sx, sy, qx, training=False, return_vis=False):
        s_fm = self.extractor(sx)
        q_fm = self.extractor(qx)
        
        if return_vis:
            s_feat, s_heads, s_scores, s_mu, s_var, s_weight = self.dlr_pool(s_fm, return_vis=True)
            q_feat, q_heads, q_scores, q_mu, q_var, q_weight = self.dlr_pool(q_fm, return_vis=True)
        else:
            s_feat, s_heads = self.dlr_pool(s_fm)
            q_feat, q_heads = self.dlr_pool(q_fm)
        
        classes = torch.unique(sy, sorted=True)
        raw = torch.stack([s_feat[sy == c].mean(0) for c in classes])
        if training: self.update_ema(s_feat, sy)
        cal = self.calibrate(raw, sy)
        final = cal + 0.05 * F.normalize(self.semantic_step[classes], dim=1)
        
        logits = F.cosine_similarity(q_feat.unsqueeze(1), final.unsqueeze(0), dim=2)
        logits = logits * TEMPERATURE
        
        if return_vis:
            return logits, final, s_heads, {
                'scores': q_scores[0].detach().cpu().numpy(),
                'mu': q_mu[0].detach().cpu().numpy(),
                'var': q_var[0].detach().cpu().numpy(),
                'weight': q_weight.detach().cpu().numpy()
            }
        if training:
            return logits, final, s_heads
        return logits, final, None

In [4]:
# Cell 4: Loss Functions
def consistency_loss(head_logits_list, sy):
    if not head_logits_list: return torch.tensor(0.0, device=sy.device)
    sigma = 0.8; device = head_logits_list[0].device
    grades = sy.view(-1, 1).float()
    classes = torch.arange(4, device=device).view(1, -1).float()
    dist = (grades - classes) ** 2
    soft_target = torch.exp(-dist / (2 * sigma**2))
    soft_target = soft_target / soft_target.sum(dim=1, keepdim=True)
    loss = sum(F.kl_div(F.log_softmax(l, dim=1), soft_target, reduction='batchmean') for l in head_logits_list)
    return loss / len(head_logits_list)

def gaussian_soft_labels(n=4, sigma=0.8):
    x = torch.arange(n).float(); L = torch.zeros(n, n)
    for i in range(n): L[i] = torch.exp(-(x - i)**2 / (2 * sigma**2))
    return L / L.sum(dim=1, keepdim=True)

LDL_LABELS = gaussian_soft_labels(N_WAY, 0.8)

def ldl_loss(logits, labels, tbl):
    return F.kl_div(F.log_softmax(logits, dim=1), tbl.to(logits.device)[labels], reduction='batchmean', log_target=False)

In [5]:
# Cell 5: Training Function with PCVW Weight Tracking
def train(model, dataset, device):
    opt = torch.optim.AdamW(model.parameters(), lr=LR_BASE, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TRAIN_EPISODES, eta_min=1e-6)
    ce = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    model.train()
    
    # 🔥 ADDED: Track PCVW weights
    log = {'ce': [], 'ldl': [], 'consist': [], 'acc': [], 
           'weight_mean': [], 'weight_max': []}
    
    t0 = time.time()
    print(f"[Device] Running on: {device}", flush=True)
    
    for ep in range(TRAIN_EPISODES):
        scale = min(1.0, ep / WARMUP)
        sx, sy, qx, qy = dataset.sample_episode(N_WAY, TRAIN_K, Q_QUERY)
        sx, sy, qx, qy = sx.to(device), sy.to(device), qx.to(device), qy.to(device)
        
        logits, protos, s_heads = model(sx, sy, qx, training=True)
        
        L_ce = ce(logits, qy)
        L_ldl = ldl_loss(logits, qy, LDL_LABELS)
        L_cons = consistency_loss(s_heads, sy)
        
        L = L_ce + LDL_LAMBDA * L_ldl + CONSISTENCY_LAMBDA * scale * L_cons

        opt.zero_grad(); L.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step(); sched.step()
        
        acc = (logits.argmax(1) == qy).float().mean().item()
        log['ce'].append(L_ce.item())
        log['ldl'].append(L_ldl.item())
        log['consist'].append(L_cons.item())
        log['acc'].append(acc)
        
        # 🔥 ADDED: Track PCVW weights (softplus(w))
        with torch.no_grad():
            weight = F.softplus(model.dlr_pool.var_weight)
            log['weight_mean'].append(weight.mean().item())
            log['weight_max'].append(weight.max().item())
        
        if ep % 100 == 0: 
            print(f"  Ep{ep:04d}|CE{L_ce.item():.4f}|Acc{acc:.4f}|{(time.time()-t0)/60:.1f}min", flush=True)
            
    print(f"  Done: {(time.time()-t0)/60:.1f}min\n", flush=True)
    return log

In [6]:
# Cell 6: Evaluation Functions
@torch.no_grad()
def evaluate_detailed(model, dataset, k_shot, device, class_names=None):
    model.eval()
    all_preds = []
    all_labels = []
    
    for _ in range(TEST_EPISODES):
        sx, sy, qx, qy = dataset.sample_episode(N_WAY, k_shot, Q_QUERY)
        sx, sy, qx, qy = sx.to(device), sy.to(device), qx.to(device), qy.to(device)
        logits, _, _ = model(sx, sy, qx, training=False)
        preds = logits.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(qy.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Calculate metrics
    cm = confusion_matrix(all_labels, all_preds, labels=range(N_WAY))
    
    # Per-class metrics
    if class_names is None:
        class_names = [str(i) for i in range(N_WAY)]
    
    report = classification_report(all_labels, all_preds, labels=range(N_WAY), 
                                   target_names=class_names,
                                   output_dict=True, zero_division=0)
    
    # Macro metrics
    macro_precision = report['macro avg']['precision']
    macro_recall = report['macro avg']['recall']
    macro_f1 = report['macro avg']['f1-score']
    
    # Cohen's Kappa
    kappa = cohen_kappa_score(all_labels, all_preds)
    
    # Per-class F1
    per_class_f1 = {}
    for i, name in enumerate(class_names):
        if name in report:
            per_class_f1[f'Class_{i}'] = report[name]['f1-score']
        else:
            try:
                per_class_f1[f'Class_{i}'] = report[str(i)]['f1-score']
            except KeyError:
                per_class_f1[f'Class_{i}'] = 0.0
    
    return {
        'confusion_matrix': cm,
        'macro_precision': macro_precision * 100,
        'macro_recall': macro_recall * 100,
        'macro_f1': macro_f1 * 100,
        'kappa': kappa,
        'per_class_f1': per_class_f1,
        'all_preds': all_preds,
        'all_labels': all_labels,
        'report': report
    }

@torch.no_grad()
def evaluate_accuracy(model, dataset, k_shot, device):
    model.eval()
    accs = []
    for _ in range(TEST_EPISODES):
        sx, sy, qx, qy = dataset.sample_episode(N_WAY, k_shot, Q_QUERY)
        sx, sy, qx, qy = sx.to(device), sy.to(device), qx.to(device), qy.to(device)
        logits, _, _ = model(sx, sy, qx, training=False)
        accs.append((logits.argmax(1) == qy).float().mean().item())
    a = np.array(accs)
    return a.mean(), 1.96 * a.std() / np.sqrt(len(a))

In [7]:
# Cell 7: Visualization Functions
def plot_confusion_matrix(cm, title, save_path, class_names=None):
    if class_names is None:
        class_names = ['Class 0', 'Class 1', 'Class 2', 'Class 3']
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[Saved] {save_path}")

def plot_training_convergence(logs_by_seed, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    def smooth(data, window=50):
        if len(data) < window:
            return data
        return np.convolve(data, np.ones(window)/window, mode='valid')
    
    colors = ['blue', 'orange', 'green', 'red', 'purple']
    
    for idx, (metric, title, ax) in enumerate([
        ('ce', 'Cross-Entropy Loss', axes[0, 0]),
        ('ldl', 'Label Distribution Learning Loss', axes[0, 1]),
        ('consist', 'Multi-Head Consistency Loss', axes[1, 0]),
        ('acc', 'Training Accuracy', axes[1, 1])
    ]):
        for seed_idx, (seed, log) in enumerate(logs_by_seed.items()):
            data = np.array(log[metric])
            if len(data) > 0:
                smoothed = smooth(data, 50)
                ax.plot(range(len(smoothed)), smoothed, color=colors[seed_idx % len(colors)], 
                       label=f'Seed {seed}', alpha=0.7, linewidth=1.5)
        
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Episode', fontsize=10)
        ax.set_ylabel('Loss' if 'Loss' in title else 'Accuracy', fontsize=10)
        ax.legend(loc='best', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[Saved] {save_path}")

def visualize_internal_representations(model, dataset, device, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    
    model.eval()
    sx, sy, qx, qy = dataset.sample_episode(N_WAY, 1, 1)
    sx, sy, qx, qy = sx.to(device), sy.to(device), qx.to(device), qy.to(device)
    
    _, _, _, vis_data = model(sx, sy, qx, training=False, return_vis=True)
    
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    img = sx[0].cpu() * std + mean
    img = img.clamp(0, 1).permute(1, 2, 0).numpy()
    
    scores = vis_data['scores'].reshape(7, 7)
    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
    scores_upsampled = F.interpolate(torch.from_numpy(scores).float().unsqueeze(0).unsqueeze(0), 
                                     size=(224, 224), mode='bilinear').squeeze().numpy()
    
    mu = vis_data['mu'][:64].reshape(8, 8, -1).mean(axis=2)
    var = vis_data['var'][:64].reshape(8, 8, -1).mean(axis=2)
    mu = (mu - mu.min()) / (mu.max() - mu.min() + 1e-8)
    var = (var - var.min()) / (var.max() - var.min() + 1e-8)
    
    weight = vis_data['weight']
    weight_normalized = (weight - weight.min()) / (weight.max() - weight.min() + 1e-8)
    
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    
    axes[0].imshow(img)
    axes[0].set_title('Original Image', fontsize=12)
    axes[0].axis('off')
    
    im1 = axes[1].imshow(scores_upsampled, cmap='hot', interpolation='bilinear')
    axes[1].set_title('Lesion Attention Map', fontsize=12)
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    
    im2 = axes[2].imshow(mu, cmap='viridis')
    axes[2].set_title('Weighted Mean (μ)', fontsize=12)
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    
    im3 = axes[3].imshow(var, cmap='plasma')
    axes[3].set_title('Weighted Variance (σ²)', fontsize=12)
    axes[3].axis('off')
    plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)
    
    axes[4].hist(weight.numpy(), bins=50, alpha=0.7, color='blue', edgecolor='black')
    axes[4].set_title('PCVW Weight Distribution', fontsize=12)
    axes[4].set_xlabel('Weight Value', fontsize=10)
    axes[4].set_ylabel('Frequency', fontsize=10)
    axes[4].grid(True, alpha=0.3)
    
    plt.tight_layout()
    save_path = os.path.join(save_dir, 'internal_representations.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[Saved] {save_path}")

def visualize_feature_activations(model, dataset, device, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    
    model.eval()
    sx, sy, qx, qy = dataset.sample_episode(N_WAY, 1, 4)
    sx, sy, qx, qy = sx.to(device), sy.to(device), qx.to(device), qy.to(device)
    
    q_fm = model.extractor(qx)
    feature_maps = q_fm.detach().cpu()
    
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    images = (qx.cpu() * std + mean).clamp(0, 1)
    
    avg_activations = feature_maps.mean(dim=1)
    avg_upsampled = F.interpolate(avg_activations.unsqueeze(1), size=(224, 224), mode='bilinear').squeeze(1)
    
    fig, axes = plt.subplots(4, 3, figsize=(12, 16))
    
    for i in range(min(4, qx.shape[0])):
        img = images[i].permute(1, 2, 0).numpy()
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f'Original Image {i+1}', fontsize=10)
        axes[i, 0].axis('off')
        
        activation = avg_upsampled[i].numpy()
        axes[i, 1].imshow(activation, cmap='hot', interpolation='bilinear')
        axes[i, 1].set_title(f'Activation Map {i+1}', fontsize=10)
        axes[i, 1].axis('off')
        
        activation_norm = (activation - activation.min()) / (activation.max() - activation.min() + 1e-8)
        heatmap = plt.cm.jet(activation_norm)[:, :, :3]
        overlay = 0.6 * img + 0.4 * heatmap
        overlay = np.clip(overlay, 0, 1)
        
        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title(f'Activation Overlay {i+1}', fontsize=10)
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    save_path = os.path.join(save_dir, 'feature_activations.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[Saved] {save_path}")

def plot_overall_confusion_matrices(detailed_results, class_names=None):
    if class_names is None:
        class_names = ['Class 0', 'Class 1', 'Class 2', 'Class 3']
    
    for k in [3, 5]:
        cm_sum = None
        count = 0
        for key, val in detailed_results.items():
            if f'k{k}' in key:
                if cm_sum is None:
                    cm_sum = val['confusion_matrix'].copy()
                else:
                    cm_sum += val['confusion_matrix']
                count += 1
        
        if cm_sum is not None:
            cm_avg = cm_sum / count
            cm_norm = cm_avg.astype('float') / cm_avg.sum(axis=1)[:, np.newaxis]
            cm_norm = np.nan_to_num(cm_norm) * 100
            
            fig, ax = plt.subplots(figsize=(8, 6))
            sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Blues',
                       xticklabels=class_names, yticklabels=class_names, ax=ax)
            ax.set_xlabel('Predicted', fontsize=12)
            ax.set_ylabel('True', fontsize=12)
            ax.set_title(f'Average Confusion Matrix - {k}-Shot (Across 5 Seeds)', fontsize=14)
            plt.tight_layout()
            save_path = os.path.join(PLOT_DIR, f'confusion_matrix_overall_k{k}.png')
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"[Saved] {save_path}")

def print_detailed_metrics_table(detailed_results):
    print("\n" + "="*100)
    print("DETAILED METRICS TABLE - DLR-PCVW")
    print("="*100)
    
    print(f"{'Seed':<6} {'Shot':<6} {'Precision (%)':<15} {'Recall (%)':<15} {'F1-Score (%)':<15} {'Kappa':<10} {'Per-Class F1':<30}")
    print("-"*100)
    
    for key, val in sorted(detailed_results.items()):
        parts = key.split('_')
        seed = parts[1] if len(parts) > 1 else '0'
        k = parts[3][1:] if len(parts) > 3 else '3'
        
        per_class_str = ""
        for i in range(4):
            class_key = f'Class_{i}'
            if class_key in val['per_class_f1']:
                f1_val = val['per_class_f1'][class_key] * 100
                per_class_str += f"C{i}:{f1_val:.1f}% "
            else:
                per_class_str += f"C{i}:N/A "
        
        print(f"{seed:<6} {k:<6} {val['macro_precision']:<15.2f} {val['macro_recall']:<15.2f} "
              f"{val['macro_f1']:<15.2f} {val['kappa']:<10.4f} {per_class_str:<30}")
    
    print("="*100)

In [8]:
# UPDATE THIS FUNCTION IN CELL 7
def visualize_internal_representations(model, dataset, device, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    
    model.eval()
    sx, sy, qx, qy = dataset.sample_episode(N_WAY, 1, 1)
    sx, sy, qx, qy = sx.to(device), sy.to(device), qx.to(device), qy.to(device)
    
    _, _, _, vis_data = model(sx, sy, qx, training=False, return_vis=True)
    
    # FIX: Properly handle the tensor dimensions
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    
    # Get the first image from the batch and denormalize
    img_tensor = sx[0:1]  # Keep batch dimension
    img_tensor = img_tensor.cpu() * std.cpu() + mean.cpu()
    img_tensor = img_tensor.clamp(0, 1)
    
    # Now remove batch dimension and permute
    img = img_tensor.squeeze(0).permute(1, 2, 0).numpy()
    
    scores = vis_data['scores'].reshape(7, 7)
    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
    scores_upsampled = F.interpolate(torch.from_numpy(scores).float().unsqueeze(0).unsqueeze(0), 
                                     size=(224, 224), mode='bilinear').squeeze().numpy()
    
    mu = vis_data['mu'][:64].reshape(8, 8, -1).mean(axis=2)
    var = vis_data['var'][:64].reshape(8, 8, -1).mean(axis=2)
    mu = (mu - mu.min()) / (mu.max() - mu.min() + 1e-8)
    var = (var - var.min()) / (var.max() - var.min() + 1e-8)
    
    weight = vis_data['weight']
    
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    
    axes[0].imshow(img)
    axes[0].set_title('Original Image', fontsize=12)
    axes[0].axis('off')
    
    im1 = axes[1].imshow(scores_upsampled, cmap='hot', interpolation='bilinear')
    axes[1].set_title('Lesion Attention Map', fontsize=12)
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    
    im2 = axes[2].imshow(mu, cmap='viridis')
    axes[2].set_title('Weighted Mean (μ)', fontsize=12)
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    
    im3 = axes[3].imshow(var, cmap='plasma')
    axes[3].set_title('Weighted Variance (σ²)', fontsize=12)
    axes[3].axis('off')
    plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)
    
    axes[4].hist(weight.numpy(), bins=50, alpha=0.7, color='blue', edgecolor='black')
    axes[4].set_title('PCVW Weight Distribution', fontsize=12)
    axes[4].set_xlabel('Weight Value', fontsize=10)
    axes[4].set_ylabel('Frequency', fontsize=10)
    axes[4].grid(True, alpha=0.3)
    
    plt.tight_layout()
    save_path = os.path.join(save_dir, 'internal_representations.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[Saved] {save_path}")

In [9]:
# Cell 8: Main Training Function (UPDATED - Saves ALL models)
def main_training():
    print("="*65, flush=True)
    print("  DLR-PCVW", flush=True)
    print(f"  FIX: Learnable w*sigma fusion (No unstable projection)", flush=True)
    print("="*65, flush=True)
    
    SPLITS = make_splits(DATA_ROOT, SPLIT_RATIO, SPLIT_SEED)
    train_cache = build_cache(SPLITS, "train", use_aug=True)
    test_cache = build_cache(SPLITS, "test", use_aug=False)
    
    all_results = {1: [], 3: [], 5: [], 10: []}
    all_detailed_results = {}
    all_logs = {}
    saved_model = None
    
    # 🔥 NEW: Create model save directory for ALL models
    MODEL_SAVE_DIR = "saved_models"
    os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
    
    class_names = [SPLITS[cid]['name'] for cid in sorted(SPLITS.keys())]
    print(f"[Classes] {class_names}", flush=True)
    
    for seed in SEEDS:
        print(f"\n{'='*45}\n  Seed {seed}\n{'='*45}", flush=True)
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        
        model = DLR_ProtoNet().to(DEVICE)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"[Model]     {trainable:,} trainable (Removed 3.2M params)", flush=True)
        print(f"[Fix]       w * sigma fusion (Softplus init -3.0)", flush=True)
        print(f"[Cache]     pre-loaded | leakage=ZERO ✓", flush=True)
        
        log = train(model, FewShotDataset(SPLITS, "train", cache=train_cache, use_aug=True), DEVICE)
        all_logs[seed] = log
        
        # 🔥 SAVE ALL MODELS (NOT JUST SEED 0)
        model_path = os.path.join(MODEL_SAVE_DIR, f'dlr_pcvw_seed{seed}_model.pth')
        torch.save(model.state_dict(), model_path)
        print(f"[Saved] Model weights for seed {seed} to {model_path}")
        
        # Keep seed 0 for visualization (same as before)
        if seed == 0:
            saved_model = model
        
        test_ds = FewShotDataset(SPLITS, "test", cache=test_cache, use_aug=False)
        
        # Evaluate accuracy for all shots
        for k in [1, 3, 5, 10]:
            acc, ci = evaluate_accuracy(model, test_ds, k, DEVICE)
            all_results[k].append(acc)
            print(f"Seed{seed}|{k:2d}-Shot:{acc*100:.2f}%±{ci*100:.2f}%", flush=True)
        
        # Detailed evaluation for ALL shots (1, 3, 5, 10)
        for k in [1, 3, 5, 10]:
            detailed = evaluate_detailed(model, test_ds, k, DEVICE, class_names)
            all_detailed_results[f'seed_{seed}_k{k}'] = detailed
            
            cm_path = os.path.join(PLOT_DIR, f'confusion_matrix_seed{seed}_k{k}.png')
            plot_confusion_matrix(
                detailed['confusion_matrix'],
                f'Confusion Matrix - Seed {seed}, {k}-Shot',
                cm_path,
                class_names
            )
        
        # Clean up GPU memory (same as before)
        if seed != 0:
            del model
            torch.cuda.empty_cache()
    
    return all_results, all_detailed_results, all_logs, saved_model

In [10]:
# Cell 9: Run Training and Collect Results
results, detailed_results, logs, vis_model = main_training()

print(f"\n{'='*65}", flush=True)
print("OVERALL RESULTS (Mean ± Std across 5 seeds):")
for k in [1, 3, 5, 10]:
    acc_array = np.array(results[k])
    print(f"4-Way{k:2d}-Shot:{acc_array.mean()*100:.2f}%±{acc_array.std()*100:.2f}%", flush=True)
print(f"\n[Done] Training Complete!", flush=True)

  DLR-PCVW
  FIX: Learnable w*sigma fusion (No unstable projection)
[Splits] seed=42 | ratio=0.7
[Cache] Loading train images...
[Cache] 967 imgs | 582MB | 77.6s

[Cache] Loading test images...
[Cache] 415 imgs | 250MB | 10.7s

[Classes] ['acne0_1024', 'acne1_1024', 'acne2_1024', 'acne3_1024']

  Seed 0
[Model]     6,929,517 trainable (Removed 3.2M params)
[Fix]       w * sigma fusion (Softplus init -3.0)
[Cache]     pre-loaded | leakage=ZERO ✓
[Device] Running on: cuda
  Ep0000|CE1.6056|Acc0.3500|0.1min
  Ep0100|CE0.3903|Acc0.9250|1.2min
  Ep0200|CE0.3043|Acc1.0000|2.4min
  Ep0300|CE0.3207|Acc1.0000|3.7min
  Ep0400|CE0.2874|Acc1.0000|4.8min
  Ep0500|CE0.2751|Acc1.0000|6.2min
  Ep0600|CE0.2791|Acc1.0000|7.4min
  Ep0700|CE0.2846|Acc1.0000|8.5min
  Ep0800|CE0.2794|Acc1.0000|9.1min
  Ep0900|CE0.2803|Acc1.0000|9.8min
  Ep1000|CE0.2804|Acc1.0000|10.5min
  Ep1100|CE0.2778|Acc1.0000|11.1min
  Done: 11.8min

[Saved] Model weights for seed 0 to saved_models\dlr_pcvw_seed0_model.pth
Seed0| 1-Sho

In [11]:
# Cell 10: Overall Confusion Matrices
SPLITS = make_splits(DATA_ROOT, SPLIT_RATIO, SPLIT_SEED)
class_names = [SPLITS[cid]['name'] for cid in sorted(SPLITS.keys())]
plot_overall_confusion_matrices(detailed_results, class_names)

[Splits] seed=42 | ratio=0.7
[Saved] Final-dlr_pcvw_224px\confusion_matrix_overall_k3.png
[Saved] Final-dlr_pcvw_224px\confusion_matrix_overall_k5.png


In [12]:
# Cell 11: Create Ensemble from All Saved Models
import os
import torch
import torch.nn.functional as F

MODEL_SAVE_DIR = "saved_models"
print("="*65)
print("  CREATING ENSEMBLE FROM ALL SAVED MODELS")
print("="*65)

def load_all_models():
    """Load all saved seed models"""
    models = []
    loaded_seeds = []
    
    for seed in SEEDS:
        model_path = os.path.join(MODEL_SAVE_DIR, f'dlr_pcvw_seed{seed}_model.pth')
        
        if os.path.exists(model_path):
            print(f"✅ Loading seed {seed} from {model_path}...")
            model = DLR_ProtoNet().to(DEVICE)
            model.load_state_dict(torch.load(model_path, map_location=DEVICE))
            model.eval()
            models.append(model)
            loaded_seeds.append(seed)
        else:
            print(f"❌ Model for seed {seed} not found at {model_path}")
    
    print(f"\n✅ Loaded {len(models)} models for seeds: {loaded_seeds}")
    return models, loaded_seeds

class EnsembleDLR:
    """Ensemble of multiple DLR models for deployment"""
    def __init__(self, models, device):
        self.models = models
        self.device = device
        self.num_models = len(models)
        print(f"✅ Ensemble initialized with {self.num_models} models")
    
    @torch.no_grad()
    def predict(self, sx, sy, qx):
        """Get ensemble prediction"""
        all_logits = []
        
        for model in self.models:
            model.eval()
            logits, _, _ = model(
                sx.to(self.device), 
                sy.to(self.device), 
                qx.to(self.device), 
                training=False
            )
            all_logits.append(logits.cpu())
        
        # Average logits
        avg_logits = torch.mean(torch.stack(all_logits), dim=0)
        return avg_logits
    
    @torch.no_grad()
    def predict_with_confidence(self, sx, sy, qx):
        """Get prediction with confidence scores"""
        avg_logits = self.predict(sx, sy, qx)
        probs = F.softmax(avg_logits, dim=1)
        confidence, preds = torch.max(probs, dim=1)
        
        return preds, confidence, probs

# Load all models
print("\nLoading ensemble models...")
ensemble_models, loaded_seeds = load_all_models()

if len(ensemble_models) > 0:
    # Create ensemble
    ensemble = EnsembleDLR(ensemble_models, DEVICE)
    
    # Save ensemble
    ensemble_path = os.path.join(MODEL_SAVE_DIR, 'ensemble_models.pt')
    torch.save({
        'models': [model.state_dict() for model in ensemble_models],
        'seed_indices': loaded_seeds,
        'num_models': len(ensemble_models)
    }, ensemble_path)
    print(f"\n✅ Ensemble saved to {ensemble_path}")
    
    print("\n" + "="*65)
    print("  ✅ MODEL SAVING COMPLETE!")
    print(f"  Total models: {len(ensemble_models)}")
    print(f"  Saved to: {MODEL_SAVE_DIR}/")
    print("="*65)
else:
    print("❌ No models found to create ensemble!")

  CREATING ENSEMBLE FROM ALL SAVED MODELS

Loading ensemble models...
✅ Loading seed 0 from saved_models\dlr_pcvw_seed0_model.pth...
✅ Loading seed 1 from saved_models\dlr_pcvw_seed1_model.pth...
✅ Loading seed 2 from saved_models\dlr_pcvw_seed2_model.pth...
✅ Loading seed 3 from saved_models\dlr_pcvw_seed3_model.pth...
✅ Loading seed 4 from saved_models\dlr_pcvw_seed4_model.pth...

✅ Loaded 5 models for seeds: [0, 1, 2, 3, 4]
✅ Ensemble initialized with 5 models

✅ Ensemble saved to saved_models\ensemble_models.pt

  ✅ MODEL SAVING COMPLETE!
  Total models: 5
  Saved to: saved_models/


In [13]:
# Cell 12: Verify All Models Saved
import os

MODEL_SAVE_DIR = "saved_models"
print("="*65)
print("  VERIFYING SAVED MODELS")
print("="*65)

# Check individual models
total_size = 0
for seed in SEEDS:
    model_path = os.path.join(MODEL_SAVE_DIR, f'dlr_pcvw_seed{seed}_model.pth')
    if os.path.exists(model_path):
        size_mb = os.path.getsize(model_path) / (1024 * 1024)
        total_size += size_mb
        print(f"✅ Seed {seed}: {size_mb:.2f} MB")
    else:
        print(f"❌ Seed {seed}: NOT FOUND")

# Check ensemble
ensemble_path = os.path.join(MODEL_SAVE_DIR, 'ensemble_models.pt')
if os.path.exists(ensemble_path):
    size_mb = os.path.getsize(ensemble_path) / (1024 * 1024)
    print(f"\n✅ Ensemble: {size_mb:.2f} MB")
else:
    print(f"\n❌ Ensemble: NOT FOUND")

print(f"\n✅ Total models size: {total_size:.2f} MB")
print("="*65)

  VERIFYING SAVED MODELS
✅ Seed 0: 26.92 MB
✅ Seed 1: 26.92 MB
✅ Seed 2: 26.92 MB
✅ Seed 3: 26.92 MB
✅ Seed 4: 26.92 MB

✅ Ensemble: 134.58 MB

✅ Total models size: 134.59 MB
